#### Auto Loader ingestion - incremental read into Bronze

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd", "Catalog")
dbutils.widgets.text("schema", "elina_sharabura", "Schema")

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("autoloader_ingestion")

from pyspark.sql.functions import col, current_timestamp

CATALOG = dbutils.widgets.get("catalog")
LOGIN = dbutils.widgets.get("schema")

SOURCE_PATH = f"/Volumes/{CATALOG}/{LOGIN}/streaming_source/events"
CHECKPOINT_PATH = f"/Volumes/{CATALOG}/{LOGIN}/streaming_source/_checkpoints/autoloader/events"
TARGET_TABLE = f"{CATALOG}.{LOGIN}_bronze.events"

df_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", CHECKPOINT_PATH)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .load(SOURCE_PATH)
)

df_bronze = (
    df_stream
        .withColumn("source_filename", col("_metadata.file_name"))
        .withColumn("ingestion_timestamp", current_timestamp())
)

query = (
    df_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
)

query.awaitTermination()

##### Streaming statistics - files per batch

In [0]:
for progress in query.recentProgress:
    source = progress["sources"][0]
    logger.info(f"Batch {progress['batchId']}: {source['numInputRows']} rows processed")

##### Schema evolution check - new column (`device_type`)

In [0]:
spark.sql(f"DESCRIBE {TARGET_TABLE}").show()
spark.sql(f"SELECT COUNT(*) FROM {TARGET_TABLE}").show()

##### Rescued data column check

In [0]:
spark.sql(f"SELECT qty, _rescued_data FROM {TARGET_TABLE} WHERE _rescued_data IS NOT NULL").show(truncate=False)

##### Schema evolution check - renamed column

In [0]:
spark.sql(f"SELECT event_type, event_category, COUNT(*) FROM {TARGET_TABLE} GROUP BY event_type, event_category").show()

##### Trigger types experiment - `availableNow` vs `once`

In [0]:
demo_checkpoint = f"/Volumes/{CATALOG}/{LOGIN}/streaming_source/_checkpoints/autoloader/trigger_experiment"
demo_table = f"{CATALOG}.{LOGIN}_bronze.events_trigger_experiment"

df_once = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", demo_checkpoint)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.maxFilesPerTrigger", 300)
        .option("header", "true")
        .load(SOURCE_PATH)
)

df_test_bronze = (
    df_once
        .withColumn("source_filename", col("_metadata.file_name"))
        .withColumn("ingestion_timestamp", current_timestamp())
)

experiment_query = (
    df_test_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", demo_checkpoint)
        .option("mergeSchema", "true")
        .trigger(once=True)
        .toTable(demo_table)
)

experiment_query.awaitTermination()

logger.info(f"Rows processed with trigger(once=True): {spark.table(demo_table).count()}")
logger.info(f"Batches: {len(experiment_query.recentProgress)}")

##### Cleanup - remove isolated test artifacts

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {demo_table}")
dbutils.fs.rm(demo_checkpoint, recurse=True)

##### Safe reload - idempotency check

In [0]:
before_count = spark.table(TARGET_TABLE).count()

reload_query = (
    df_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
)
reload_query.awaitTermination()

after_count = spark.table(TARGET_TABLE).count()

logger.info(f"Before: {before_count}")
logger.info(f"After:  {after_count}")